## In-Class Lecture 33-1 - How Far Is a Hand-Written Loop from BLAS?

In this lecture, we will write the same matrix multiply three ways, prove that all three produce the same answer, and measure how much the order of memory access and a tuned library change the run time. Homework 9 builds directly on the code you write today.

### Reviewing Opening your Environment on the Notre Dame Machines

<b>1.</b> Log into the Notre Dame Machines as you performed on Homework 1 using Tabby Terminal 

Open Tabby Terminal on your machine.

Type in: <code>ssh -Y username@student13.cse.nd.edu</code>
<ul><li>"username" is your NetID.</li>
<ul>
<li>Example: my email is matt.morrison@nd.edu, but my NetID is mmorri22, so I would enter:<p>
<code>ssh -Y mmorri22@student13.cse.nd.edu</code><p>
The <code>-Y</code> is included to treat the Notre Dame machines as a trusted client<p>
</li>
</ul>
</ul>

2. Next, we will create a directory where you will save your In-Class Code. First, we will go into the Dropbox directory where you will submit your code:

<code>cd esc-courses</code>

<code>cd fa26-cse-20133.01</code>

<code>cd dropbox</code>

Make a directory for you to practice your code for this lecture

    mkdir inclass33
    cd inclass33

Enable script so you can run the most up-to-date versions of C and C++:

    scl enable gcc-toolset-15 bash

> <b>Note:</b> If you use the <code>-std=c2x</code> and get the following error, this means you did not run the enable command<br>
> <code>gcc: error: unrecognized command line option ‘-std=c2x’; did you mean ‘-std=c1x’?</code>

To obtain the code for today's lecture, perform the following commands:

    wget https://raw.githubusercontent.com/mmorri22/cse20133/main/inclass/lec33/setup.sh
    chmod a+rx setup.sh
    ./setup.sh
    
To confirm, perform the <code>ls</code> command. You will see the following:

    Makefile  matmul.c

### In-Class Reading Discussion

<font color = "red">Class Introduction Question #1 - What is machine epsilon, and what happens when you add a value much smaller than epsilon to 1.0?</font>

<font color = "red">Class Introduction Question #2 - The Patriot's error per tick was about 0.000000095 seconds. How did such a small error become large enough to miss a missile?</font>

<font color = "red">Class Introduction Question #3 - Why does the <code>float</code> accumulator drift so much more than the <code>double</code> accumulator, and why does the integer tick count not drift at all?</font>

<font color = "red">Class Introduction Question #4 - C stores matrices in row-major order. Explain why summing by columns is so much slower than summing by rows, even though the arithmetic is identical.</font>

<font color = "red">Class Introduction Question #5 - Why should you use <code>CLOCK_MONOTONIC</code> and report the median of several runs when timing code?</font>

<font color = "red">Class Introduction Question #6 - What is the difference between the BLAS <i>interface</i> and a BLAS <i>implementation</i>, and why does that difference matter for flight software that may run on different processors?</font>

### What Is Already Written for You

<code>matmul.c</code> multiplies two 512 x 512 matrices of <code>double</code>. These functions are complete:
<ul>
    <li><code>now_seconds()</code> - the monotonic timer from Reading 33-2</li>
    <li><code>fill_matrix()</code> - fills a matrix with repeatable values between -1.0 and 1.0</li>
    <li><code>max_abs_diff()</code> - the largest difference between two matrices, element by element</li>
    <li><code>gflops()</code> - converts a run time into GFLOP/s using 2M<sup>3</sup> operations</li>
</ul>

Every matrix is stored in one block from <code>malloc</code>, so element <code>[row][col]</code> is <code>mat[row * M + col]</code>. The starter compiles and runs as-is, but it does nothing yet.

### Step 1 - The Textbook Loop: <code>matmul_ijk</code>

Write the triple loop exactly as you learned matrix multiplication in math class. Each element of <code>C</code> is the dot product of a row of <code>A</code> and a column of <code>B</code>:

<p align="center">c<sub>ij</sub> = &Sigma;<sub>k</sub> a<sub>ik</sub> b<sub>kj</sub></p>

Loop over <code>i</code>, then <code>j</code>, then <code>k</code>. Delete the <code>(void)</code> line when you are done.

### Step 2 - Time It

In <code>main</code>, record <code>now_seconds()</code>, call <code>matmul_ijk(a, b, c_ijk)</code>, and subtract to get the elapsed time. Print the time and the GFLOP/s.

    make matmul

<b>Before you go on:</b> in the innermost loop, which matrix is being read one row at a time, and which is being read one <i>column</i> at a time? Based on Reading 33-2, which one is hurting you?

### Step 3 - Swap Two Loops: <code>matmul_ikj</code>

Now write the same multiply with the loops in <code>i</code>, <code>k</code>, <code>j</code> order. Inside the <code>k</code> loop, look up <code>a[i * M + k]</code> once, then loop over <code>j</code> and add <code>a_ik * b[k * M + j]</code> into <code>c[i * M + j]</code>.

Since this version <i>adds into</i> <code>c</code>, it depends on <code>c_ikj</code> starting at zero. Find where that already happens in <code>main</code>.

### Step 4 - Time It and Check It

Time <code>matmul_ikj(a, b, c_ikj)</code> the same way. Print its time, its GFLOP/s, its speedup over <code>i-j-k</code>, and <code>max_abs_diff(c_ijk, c_ikj)</code>.

### Step 5 - Call BLAS

Replace your loops with one function call, using the argument order from Reading 33-2:

    cblas_dgemm(CblasRowMajor, CblasNoTrans, CblasNoTrans, M, M, M,
    	    1.0, a, M, b, M, 0.0, c_blas, M);

Time it, and print its time, GFLOP/s, speedup over <code>i-j-k</code>, and <code>max_abs_diff(c_ijk, c_blas)</code>.

### Step 6 - Validate

Never compare floating-point results with <code>==</code>. For each comparison, print <code>PASS</code> if the maximum difference is at most <code>TOLERANCE</code> and <code>FAIL</code> otherwise.

### Discussion

Once your program prints all three results, answer these with your neighbors:
<ol>
    <li>How much faster was <code>i-k-j</code> than <code>i-j-k</code>? You performed <i>exactly</i> the same multiplications and additions. Where did the time go?</li>
    <li>Look at the maximum difference for <code>i-k-j</code>. Is it exactly zero? Look at the order in which the terms of each <code>c<sub>ij</sub></code> are added in both versions, and explain the result.</li>
    <li>BLAS probably did <i>not</i> match to the last bit. Is BLAS wrong? Which of the three answers is "correct"?</li>
    <li>Run <code>make matmul</code> two or three more times. Do your times change? What does that tell you about reporting a single measurement in Homework 9?</li>
</ol>

<b>Time Permitting:</b> Change <code>#define M 512</code> to 1024 and run again. The amount of work grew by a factor of 8. Did every version's time grow by a factor of 8?

## Compilation Run

    make matmul

When you are finished, all three versions should print a time and a GFLOP/s, and both comparisons should print <code>PASS</code>.